# import libs

In [27]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder
import numpy as np
from sklearn.feature_selection import RFECV
from sklearn.model_selection import StratifiedKFold
import lightgbm as lgb
import optuna
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
import numpy as np
import joblib
import json
import os
from scipy import stats

# load model and params

In [28]:
# Load artifacts 
best_model = joblib.load("D:/credit-card-crosssell/models/xgb_propensity.pkl")
encoders   = joblib.load("D:/credit-card-crosssell/models/encoders.pkl")

with open("D:/credit-card-crosssell/models/selected_features.json") as f:
    selected = json.load(f)

with open("D:/credit-card-crosssell/models/best_params.json") as f:
    best_params = json.load(f)

with open("D:/credit-card-crosssell/models/model_results.json") as f:
    results = json.load(f)

print(f"Selected features : {len(selected)}")
print(f"AUC Test          : {results['auc_test']}")
print(f"AUC OOT           : {results['auc_oot']}")

Selected features : 27
AUC Test          : 0.9033
AUC OOT           : 0.9042


In [29]:
# load oot label
label_oot = pd.read_csv("D:/credit-card-crosssell/data/result_oot.csv")
y_proba = label_oot['propensity_score']
y_oot = label_oot['label']
# Load current data 
curr = pd.read_csv("D:/credit-card-crosssell/data/feature_store_current.csv")

drop_cols  = ['customer_id', 'observation_date']
X_curr     = curr.drop(columns=drop_cols)

# Encode 
X_curr_encoded = X_curr.copy()
for col in ['gender', 'province']:
    X_curr_encoded[col] = encoders[col].transform(X_curr[col])

# Score 
scores = best_model.predict_proba(X_curr_encoded[selected])[:, 1]

result = pd.DataFrame({
    'customer_id'     : curr['customer_id'],
    'observation_date': curr['observation_date'],
    'propensity_score': scores.round(4),
}).sort_values('propensity_score', ascending=False).reset_index(drop=True)

In [30]:
# Apply lên current data
y_curr_pred = (scores >= 0.24).astype(int)

print(f"\nKH predict = 1 (contact) : {y_curr_pred.sum():,}")
print(f"KH predict = 0 (skip)    : {(y_curr_pred == 0).sum():,}")


KH predict = 1 (contact) : 4,739
KH predict = 0 (skip)    : 261


In [31]:
# Apply threshold lên current data
y_curr_pred = (scores >= 0.24).astype(int)

curr_result = pd.DataFrame({
    'customer_id'     : curr['customer_id'],
    'observation_date': curr['observation_date'],
    'propensity_score': scores.round(4),
    'predicted_label' : y_curr_pred,
})

print(f"KH predict = 1 (contact) : {y_curr_pred.sum():,}")
print(f"KH predict = 0 (skip)    : {(y_curr_pred == 0).sum():,}")
print(f"\nScore distribution của predicted = 1:")
print(curr_result[curr_result['predicted_label']==1]['propensity_score'].describe())

KH predict = 1 (contact) : 4,739
KH predict = 0 (skip)    : 261

Score distribution của predicted = 1:
count    4739.000000
mean        0.744386
std         0.231421
min         0.240000
25%         0.529950
50%         0.816300
75%         0.972900
max         0.995600
Name: propensity_score, dtype: float64


# test design and sample size

test "campaign có hiệu quả không" và test cả "model có thực sự add value không".


### Bốn arms và mục đích

| Arm | Predict | Contact | Mục đích |
|---|---|---|---|
| A | 1 | Yes | Model + campaign → ground truth của strategy |
| B | 1 | No | Model value thuần — không có campaign effect |
| C | 0/1 | No | Natural baseline — không làm gì |
| D | 0 | Yes | Random contact — campaign không có model |

**Test 1: A vs C** — Model + strategy có hiệu quả hơn không làm gì không?

```
H₀: p_A - p_C ≤ 0
→ Đây là overall business value của toàn bộ system
→ Câu hỏi quan trọng nhất với business
```

**Test 2: A vs B** — Contact có add value lên trên model không?

```
H₀: p_A - p_B ≤ 0
→ Isolate campaign uplift
→ Nếu A ≈ B → KH predicted = 1 sẽ tự convert dù không contact
→ Campaign waste money
```

**Test 3: B vs C** — Model predict = 1 có tự convert cao hơn không?

```
H₀: p_B - p_C ≤ 0
→ Đây là test model discrimination power
→ Nếu B > C → model đang identify đúng high-propensity KH
→ Quan trọng để validate model chứ không chỉ campaign
```

**Test 4: A vs D** — Model có tốt hơn random contact không?

```
H₀: p_A - p_D ≤ 0
→ Đây là business case của model
→ Nếu A ≈ D → model không add value, contact random cũng được
→ Nếu A > D → model worth the investment
```

**Test 5: D vs C** — Gọi bừa có hiệu quả hơn không làm gì không?

```
H₀: p_D - p_C ≤ 0
→ Test xem campaign channel itself có value không
→ Tách biệt channel effect khỏi model effect
```

## sample size

In [32]:
def sample_size(z_alpha, z_beta, p_bar, r, delta):
    """
    z_alpha : Z_{1- alpha} — từ mức ý nghĩa alpha (Type I error)
              alpha = 0.05 → z_alpha = 1.645 (one-sided)
    z_beta  : Z_{1-β} — từ power (1 - Type II error)
              power = 0.80 → z_beta = 0.842
    p_bar   : pooled conversion rate = (p_A + p_C) / 2
    r       : allocation ratio = n_A / n_C
              r = 2 → nhóm A gấp đôi nhóm C
    delta   : MDE = p_A - p_C
              delta = 0.05 → detect được effect ≥ 5%
    
    return  : n_C (cỡ mẫu nhóm nhỏ hơn)
              n_A = r * n_C
    """
    n = ((z_alpha + z_beta)**2 * p_bar * (1 - p_bar) * (r + 1)) / (r * delta**2)
    return int(np.ceil(n))

1. p_C baseline là bao nhiêu? → lấy từ historical data
2. MDE cho từng comparison là bao nhiêu?
3. Allocation ratio giữa 4 arms như thế nào?
   Equal (25/25/25/25) hay unequal?
4. Primary test là gì? → quyết định α

Chọn test strategy phân cấp priority:

```
Primary test   : A vs C (overall value)     → α = 0.05
Secondary tests: A vs B, B vs C, A vs D    → α = 0.025 (Bonferroni/2)
Exploratory    : D vs C                    → α = 0.10
```

In [33]:
# Primary test: A vs C
label_historical = pd.read_csv("D:/credit-card-crosssell/data/label_historical.csv")
p_C     = label_historical['converted'].mean()
p_A     = p_C + 0.05
p_bar   = (p_A + p_C) / 2
r       = 2           # n_A / n_C = 40% / 20%
delta   = p_A - p_C
alpha   = 0.05
power   = 0.80

z_alpha = stats.norm.ppf(1 - alpha)   # 1.645
z_beta  = stats.norm.ppf(power)       # 0.842

n_C = sample_size(z_alpha, z_beta, p_bar, r, delta)
n_A = r * n_C
n_B = n_C
n_D = n_C

total = n_A + n_B + n_C + n_D

In [34]:
print(f"z_alpha : {z_alpha:.3f}")
print(f"z_beta  : {z_beta:.3f}")
print(f"p_bar   : {p_bar:.4f}")
print(f"delta   : {delta:.4f}")
print(f"r       : {r}")
print()
print(f"n_C (control)          : {n_C:,}")
print(f"n_A (model + contact)  : {n_A:,}")
print(f"n_B (model, no contact): {n_B:,}")
print(f"n_D (random contact)   : {n_D:,}")
print(f"Total                  : {total:,}")
print(f"Current pool           : 5,000")
print(f"Sufficient             : {'Yes' if 5000 >= total else 'No'}")

z_alpha : 1.645
z_beta  : 0.842
p_bar   : 0.7361
delta   : 0.0500
r       : 2

n_C (control)          : 721
n_A (model + contact)  : 1,442
n_B (model, no contact): 721
n_D (random contact)   : 721
Total                  : 3,605
Current pool           : 5,000
Sufficient             : Yes


In [35]:
np.random.seed(42)

# Pool predict=1 → candidates cho Arm A và B
pool_1 = curr_result[curr_result['predicted_label'] == 1]['customer_id'].values
# Pool predict=0 → candidates cho Arm D
pool_0 = curr_result[curr_result['predicted_label'] == 0]['customer_id'].values

In [36]:
# Sample từng arm 

# Arm A: model + contact, predict=1, n=1442
arm_A = np.random.choice(pool_1, n_A, replace=False)

# Arm B: model no contact, predict=1, n=721
# Lấy từ pool_1 sau khi đã loại arm_A
remaining_1 = np.setdiff1d(pool_1, arm_A)
arm_B = np.random.choice(remaining_1, n_B, replace=False)

# Arm C: control, không contact, n=721
# Lấy từ toàn bộ pool (cả predict=0 và 1 chưa được assign)
remaining_all = np.setdiff1d(curr_result['customer_id'].values, 
                              np.concatenate([arm_A, arm_B]))
arm_C = np.random.choice(remaining_all, n_C, replace=False)

# Arm D: random contact, predict=0, n=721
remaining_0 = np.setdiff1d(pool_0, arm_C)
if len(remaining_0) >= n_D:
    arm_D = np.random.choice(remaining_0, n_D, replace=False)
else:
    # Nếu pool_0 không đủ thì lấy thêm từ remaining_all
    arm_D = np.random.choice(
        np.setdiff1d(remaining_all, arm_C), n_D, replace=False)

In [42]:
print(f"Arm A (model + contact)  : {len(arm_A):,}")
print(f"Arm B (model, no contact): {len(arm_B):,}")
print(f"Arm C (control)          : {len(arm_C):,}")
print(f"Arm D (random contact)   : {len(arm_D):,}")
print(f"Total                    : {len(arm_A)+len(arm_B)+len(arm_C)+len(arm_D):,}")

# Overlap check
all_ids = np.concatenate([arm_A, arm_B, arm_C, arm_D])
print(f"Unique KH                : {len(np.unique(all_ids)):,}")
print(f"Overlap                  : {'None' if len(np.unique(all_ids)) == len(all_ids) else 'Has overlap'}")

Arm A (model + contact)  : 1,442
Arm B (model, no contact): 721
Arm C (control)          : 721
Arm D (random contact)   : 721
Total                    : 3,605
Unique KH                : 3,605
Overlap                  : None


In [43]:
# export then claude gen label
# Tạo assignment table
assignment = pd.DataFrame({
    'customer_id': np.concatenate([arm_A, arm_B, arm_C, arm_D]),
    'arm'        : (['A'] * len(arm_A) + 
                    ['B'] * len(arm_B) + 
                    ['C'] * len(arm_C) + 
                    ['D'] * len(arm_D))
})

# Merge với propensity scores
assignment = assignment.merge(
    curr_result[['customer_id', 'propensity_score', 'predicted_label']],
    on='customer_id'
)

print(assignment.groupby('arm').agg(
    n                = ('customer_id', 'count'),
    avg_score        = ('propensity_score', 'mean'),
    pct_predicted_1  = ('predicted_label', 'mean')
).round(4))

# Save
assignment.to_csv(
    "D:/credit-card-crosssell/data/campaign_assignment.csv", 
    index=False
)
print("\nSaved: campaign_assignment.csv")

        n  avg_score  pct_predicted_1
arm                                  
A    1442     0.7505           1.0000
B     721     0.7531           1.0000
C     721     0.6781           0.9098
D     721     0.6897           0.9168

Saved: campaign_assignment.csv


## ab test

In [44]:
outcome = pd.read_csv("D:\credit-card-crosssell\data\campaign_outcomes.csv")

In [45]:
outcome

,customer_id,arm,propensity_score,predicted_label,contacted,contact_date,label_window_start,label_window_end,converted,conversion_date
0,KH022435,A,0.9165,1,1,2025-07-07,2025-07-01,2025-09-30,1,2025-07-15
1,KH020376,A,0.9594,1,1,2025-07-11,2025-07-01,2025-09-30,1,2025-07-21
2,KH021465,A,0.9143,1,1,2025-07-07,2025-07-01,2025-09-30,1,2025-09-25
3,KH021939,A,0.9674,1,1,2025-07-11,2025-07-01,2025-09-30,1,2025-07-24
4,KH022970,A,0.9839,1,1,2025-07-03,2025-07-01,2025-09-30,1,2025-07-02
...,...,...,...,...,...,...,...,...,...,...
3600,KH022534,D,0.3652,1,1,2025-07-04,2025-07-01,2025-09-30,1,2025-07-30
3601,KH022559,D,0.4303,1,1,2025-07-14,2025-07-01,2025-09-30,1,2025-07-14
3602,KH022176,D,0.7172,1,1,2025-07-04,2025-07-01,2025-09-30,1,2025-07-24
3603,KH021023,D,0.5912,1,1,2025-07-12,2025-07-01,2025-09-30,1,2025-09-15


**Test 1: A vs C** — Model + strategy có hiệu quả hơn không làm gì không?

```
H₀: p_A - p_C ≤ 0
```


**Test 2: A vs B** — Contact có add value lên trên model không?

```
H₀: p_A - p_B ≤ 0
```



**Test 3: B vs C** — Model predict = 1 có tự convert cao hơn không?

```
H₀: p_B - p_C ≤ 0
```


**Test 4: A vs D** — Model có tốt hơn random contact không?

```
H₀: p_A - p_D ≤ 0
```


**Test 5: D vs C** — Gọi bừa có hiệu quả hơn không làm gì không?

```
H₀: p_D - p_C ≤ 0
```